# 総研大 2024年度 講義「シミュレーション天文学」 / SOKEN-dai FY2024 Lecture Course: “Simulation Astronomy”
### 担当：馬場淳一 (国立天文台 科学研究部/鹿児島大学 天の川銀河研究センター) / Junichi Baba (D. of Science, NAOJ / Kagoshima University)

This code is provided for educational purposes only. Users are encouraged to utilize it at their own risk. I take no responsibility for any results obtained from research or analyses conducted using this code. It is essential to deepen your understanding and verify the results independently. Additionally, it is recommended to cross-reference with other reliable resources to ensure the appropriate methodologies are applied.

このコードは教育目的で提供されています。使用にあたっては、自己責任で行ってください。提供されたコードを基に行った研究や解析によって得られた結果について、私は一切の責任を負いません。使用する際には、必ず自身の理解を深め、結果を検証してください。また、他の信頼できるリソースと照らし合わせて、適切な方法論を選択することをお勧めします。

## 学習目標と計算上の前提

G=M=1 の固定中心力場を運動する単位質量の試験粒子です。energy は比エネルギー E=v²/2−GM/r、全手法の誤差は |E(t)−E(0)| に統一しています。円軌道ではエネルギー誤差に特別な相殺があり、位置誤差や位相誤差の収束次数も調べます。固定刻みLeapfrogのシンプレクティック性は真のエネルギーの厳密保存を意味しません。r=0 の衝突軌道は対象外です。

検証課題と判定基準は [演習ガイド](../docs/exercises.md) を参照してください。


円軌道の誤差図では、解析解 $(\cos T,\sin T)$ に対する位置誤差、エネルギー誤差、実行時間を並べます。円軌道では特別な相殺により、エネルギー誤差の傾きが解法の次数より高くなることがあります。位置誤差の参照線 $h,h^2,h^4$ は傾きの目安で、誤差の大きさを予言する線ではありません。次数は漸近領域で複数の刻みの組から評価してください。ABM2では位相誤差の符号反転による相殺で、途中の刻み範囲の傾きが不規則になる場合があります。


# Kepler Problem Explanation

### Objective

The **Kepler problem** concerns the motion of a small body (such as a planet or satellite) under the gravitational influence of a central massive body (like the Sun). The problem is governed by Newton’s laws of motion and gravitation, leading to elliptical orbits, as described by Kepler’s laws.

In this case, we are solving the **two-body problem** where the central mass $M$ is fixed (assumed to be much larger than the orbiting body). The smaller body experiences a gravitational force, and we calculate its motion based on this force.

### Equations of Motion

The governing equations for the Kepler problem come from **Newton’s law of universal gravitation** and **Newton’s second law of motion**. The gravitational force exerted on the orbiting body is:

$$
\mathbf{F} = -\frac{GMm}{r^3} \mathbf{r}
$$

where:

- $G$ is the gravitational constant,
- $M$ is the mass of the central body,
- $\mathbf{r}$ is the position vector of the orbiting body,
- $r = |\mathbf{r}|$ is the magnitude of the position vector (distance from the center).

This force causes an acceleration:

$$
\mathbf{a} = \frac{\mathbf{F}}{m} = -\frac{GM}{r^3} \mathbf{r}
$$

### Reducing the Kepler Problem to a System of First-Order ODEs

To solve the Kepler problem numerically, we need to convert the second-order differential equation for motion into a system of **first-order ordinary differential equations (ODEs)**.

#### Original Equation (Second-Order ODE):

Newton’s second law provides the equation of motion:

$$
\frac{d^2 \mathbf{r}}{dt^2} = -\frac{GM}{r^3} \mathbf{r}
$$

This is a **second-order ODE** because it involves the second derivative of position with respect to time.

#### Converting to First-Order ODEs:

We introduce a new variable $\mathbf{v}$ to represent the velocity:

$$
\mathbf{v} = \frac{d\mathbf{r}}{dt}
$$

Now, we can rewrite the second-order equation as a **system of first-order ODEs**:

1. The first equation describes the time evolution of the position:
   $$
   \frac{d\mathbf{r}}{dt} = \mathbf{v}
   $$

2. The second equation describes the time evolution of the velocity, which comes from the acceleration due to gravity:
   $$
   \frac{d\mathbf{v}}{dt} = -\frac{GM}{r^3} \mathbf{r}
   $$

Here, $r = |\mathbf{r}| = \sqrt{x^2 + y^2}$ is the magnitude of the position vector.

#### System of First-Order ODEs:

Thus, the Kepler problem is reduced to the following system of four first-order ODEs:

$$
\frac{dx}{dt} = v_x
$$

$$
\frac{dy}{dt} = v_y
$$

$$
\frac{dv_x}{dt} = -\frac{GM}{r^3} x
$$

$$
\frac{dv_y}{dt} = -\frac{GM}{r^3} y
$$


### Summary:

To numerically solve the Kepler problem, we convert the second-order equation of motion into a system of first-order ODEs, which can then be solved using standard numerical integration techniques.

In [ ]:
import numpy as np
import time 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
parameters = {
    'axes.labelsize': 26,           # Set the font size of the x-axis and y-axis labels to 26
    'xtick.labelsize': 24,          # Set the font size of the x-axis tick labels to 24
    'ytick.labelsize': 24,          # Set the font size of the y-axis tick labels to 24
    'axes.titlesize': 24,           # Set the font size of the axis titles (not the overall graph title) to 24
    'figure.titlesize': 24,         # Set the font size of the overall graph title to 24
    'legend.fontsize': 22,          # Set the font size of the legend to 22
    'figure.figsize': [10, 6],      # Set the default figure size (width 10 inches, height 6 inches)
    'axes.linewidth': 2.0           # Set the linewidth of the axes
}
plt.rcParams.update(parameters)

QUICK_DEMO = False  # True: long-time examples use 1/10 of the duration


In [ ]:
G = 1
M = 1
def gravity_acceleration(r):
    """
    Compute the gravitational acceleration due to the central mass M=1.
    
    Parameters:
        r (np.ndarray): Position vector [x, y]
    
    Returns:
        a (np.ndarray): Gravitational acceleration vector [-x/r^3, -y/r^3]
    """
    r_norm = np.linalg.norm(r)  # Magnitude of the position vector
    a = - G* M * r / r_norm**3  # Gravitational acceleration
    return a

def total_energy(r, v):
    """
    Compute the total energy (kinetic + potential) for the Kepler problem.

    Parameters:
        r (np.ndarray): Position vector [x, y]
        v (np.ndarray): Velocity vector [vx, vy]
    
    Returns:
        float: Total energy E = T + U
    """
    kinetic_energy = 0.5 * np.dot(v, v)  # T = 1/2 v^2
    potential_energy = -G * M / np.linalg.norm(r)  # U = -1/|r|
    return kinetic_energy + potential_energy


METHOD_STYLES = {
    'Euler': dict(color='#000000', marker='o', linestyle='-'),
    'Heun': dict(color='#E69F00', marker='s', linestyle='--'),
    'Midpoint': dict(color='#56B4E9', marker='^', linestyle='-.'),
    'RK4': dict(color='#009E73', marker='D', linestyle=':'),
    'Adams2': dict(color='#0072B2', marker='v', linestyle='--'),
    'Adams4': dict(color='#D55E00', marker='P', linestyle='-.'),
    'Leapfrog': dict(color='#CC79A7', marker='X', linestyle='-'),
}
def compare_circular_orbit(methods):
    """For G=M=1 and radius=1, the exact position is (cos(t), sin(t))."""
    T = 10.0
    step_sizes = 0.1 / 2.0**np.arange(8)
    fig, axes = plt.subplots(1, 3, figsize=(21, 6))
    for name, method in methods.items():
        hs, position, energy, seconds = [], [], [], []
        for h in step_sizes:
            start = time.perf_counter()
            t, r, v, de = method(np.array([1., 0.]), np.array([0., 1.]), 0., T, h)
            seconds.append(time.perf_counter() - start)
            hs.append(t[1] - t[0])
            position.append(np.linalg.norm(r[-1] - [np.cos(T), np.sin(T)]))
            energy.append(de[-1])
        for ax, values in zip(axes, [position, energy, seconds]):
            # Exact zeros cannot be displayed on logarithmic axes.
            ax.loglog(hs, np.where(np.array(values) > 0, values, np.nan),
                      label=name, linewidth=2, markersize=6, **METHOD_STYLES[name])
    ref_h = np.array([step_sizes[-1], step_sizes[0]])
    for p, scale, style in [(1, 1., '--'), (2, 1., '-.'), (4, 1., ':')]:
        axes[0].loglog(ref_h, scale * ref_h**p, style, color='0.5', label=f'h^{p} (guide)')
    labels = [('Position error at T=10', 'Position error'),
              ('Energy error (not method order)', '|E(T) - E(0)|'),
              ('Measured run time', 'Seconds')]
    for ax, (title, ylabel) in zip(axes, labels):
        ax.set(xlabel='Actual step h', ylabel=ylabel, title=title)
        ax.grid(True, which='both', alpha=.4)
        ax.legend(fontsize=10)
    fig.tight_layout()
    plt.show()


## Explanation of the `euler_method` Function

This function implements the **Forward Euler method** for solving the **Kepler problem**, which involves calculating the motion of a body under gravitational influence. In this case, the position and velocity vectors of a body are updated over time using the explicit Euler method, and energy errors are tracked to evaluate the accuracy of the method.

### Function Parameters:
- **`r0` (np.ndarray)**: The initial position vector $[x_0, y_0]$, representing the starting coordinates of the body.
- **`v0` (np.ndarray)**: The initial velocity vector $[v_{x0}, v_{y0}]$, representing the initial velocity components of the body.
- **`t0` (float)**: The starting time of the simulation.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization, determining the interval between updates.

### Returns:
- **`t_values` (np.ndarray)**: An array containing the time points from $ t_0 $ to $ t_{\text{end}} $.
- **`r_values` (np.ndarray)**: An array containing the position vectors $[x, y]$ at each time point.
- **`v_values` (np.ndarray)**: An array containing the velocity vectors $[v_x, v_y]$ at each time point.
- **`energy_errors` (np.ndarray)**: An array containing the errors in the specific mechanical energy at each time point, showing how energy conservation is violated by the Euler method.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps $ N $ is calculated based on the total simulation time and step size.
   - The `t_values` array holds the time points, `r_values` stores the position vectors, and `v_values` stores the velocity vectors.
   - The initial position $ r_0 $ and velocity $ v_0 $ are assigned to the first elements of `r_values` and `v_values`.

2. **Initial Energy Calculation**:
   - The initial specific mechanical energy of the system is calculated using the `total_energy` function and stored in `initial_energy`.
   - The energy error array `energy_errors` is initialized, with the error set to zero at the initial time step.

3. **Main Loop for Time Stepping**:
   - For each time step $ n $, the position and velocity vectors at time $ t_n $ are updated using the explicit **Euler method**:
   
   **Position Update**:
   - The position is updated based on the current velocity:
     $$
     r_{n+1} = r_n + h \cdot v_n
     $$
   
   **Velocity Update**:
   - The velocity is updated based on the gravitational acceleration $ a(r_n) $, computed by the `gravity_acceleration` function:
     $$
     v_{n+1} = v_n + h \cdot a(r_n)
     $$

4. **Energy Calculation**:
   - After updating the position and velocity, the specific mechanical energy at each step is calculated using the `total_energy` function. The energy error at each step is computed as the absolute difference between the current energy and the initial energy:
     $$
     \text{energy\_error}_{n+1} = |E_{n+1} - E_0|
     $$

5. **Final Output**:
   - The function returns the `t_values` (time points), `r_values` (position vectors), `v_values` (velocity vectors), and `energy_errors` (energy errors at each time point).

### Example Workflow:
1. **Time Stepping**: The simulation progresses by updating both the position and velocity at each time step using the Euler method.
2. **Energy Error Tracking**: At each time step, the energy error is calculated, which provides insight into how well the method conserves energy.
3. **Kepler Problem Simulation**: This implementation simulates the motion of a body in a gravitational field using the Forward Euler method, which is simple but can have large energy errors due to its explicit nature.

### Key Advantages:
- **Simple Implementation**: The Forward Euler method is easy to implement and computationally cheap.
- **Direct Calculations**: The method updates position and velocity directly without the need for solving implicit equations.

### Limitations:
- **Low Accuracy and Energy Conservation**: The Forward Euler method does not conserve energy well, especially for long-term simulations. This is reflected in the `energy_errors` array.
- **Stability Issues**: For problems like the Kepler problem, the Forward Euler method may become unstable unless the step size $ h $ is sufficiently small.

In [ ]:
# Define the Forward Euler Method Function
def euler_method(r0, v0, t0, t_end, h):
    """
    Solve the Kepler problem using the explicit Forward Euler method.

    Parameters:
        r0 (np.ndarray): Initial position vector [x0, y0]
        v0 (np.ndarray): Initial velocity vector [vx0, vy0]
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size

    Returns:
        t_values (np.ndarray): Array of time points
        r_values (np.ndarray): Array of position vectors at each time point
        v_values (np.ndarray): Array of velocity vectors at each time point
        energy_errors (np.ndarray): Array of energy errors at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    
    r_values = np.zeros((N + 1, 2))  # Store position vectors [x, y]
    v_values = np.zeros((N + 1, 2))  # Store velocity vectors [vx, vy]
    
    r_values[0] = r0
    v_values[0] = v0

    # Calculate initial energy
    initial_energy = total_energy(r0, v0)
    energy_errors = np.zeros(N + 1)
    energy_errors[0] = 0  # Initial energy error is 0

    for n in range(N):
        r_n = r_values[n]
        v_n = v_values[n]
        
        # Euler method: update position and velocity
        r_values[n + 1] = r_n + h * v_n  # Update position
        v_values[n + 1] = v_n + h * gravity_acceleration(r_n)  # Update velocity

        # Calculate energy at this step
        energy_n = total_energy(r_values[n + 1], v_values[n + 1])
        energy_errors[n + 1] = abs(energy_n - initial_energy)  # Absolute error in energy

    return t_values, r_values, v_values, energy_errors

In [ ]:
# Circular-orbit demonstration: compare position and energy errors separately.
compare_circular_orbit({'Euler': euler_method})


## Explanation of the `rk2_heun_method` Function

This function implements **Heun's method**, also known as the **improved Euler method** (a 2nd-order Runge-Kutta method), to solve the **Kepler problem**. It updates the position and velocity of a body under gravitational influence and tracks energy errors, improving the accuracy over the basic Forward Euler method by using a predictor-corrector approach.

### Function Parameters:
- **`r0` (np.ndarray)**: The initial position vector $[x_0, y_0]$, representing the starting coordinates of the body.
- **`v0` (np.ndarray)**: The initial velocity vector $[v_{x0}, v_{y0}]$, representing the initial velocity components of the body.
- **`t0` (float)**: The starting time of the simulation.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization, determining the interval between updates.

### Returns:
- **`t_values` (np.ndarray)**: An array containing the time points from $ t_0 $ to $ t_{\text{end}} $.
- **`r_values` (np.ndarray)**: An array containing the position vectors $[x, y]$ at each time point.
- **`v_values` (np.ndarray)**: An array containing the velocity vectors $[v_x, v_y]$ at each time point.
- **`energy_errors` (np.ndarray)**: An array containing the errors in the specific mechanical energy at each time point, showing how well the method conserves energy.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps $ N $ is calculated based on the total simulation time and step size.
   - The `t_values` array holds the time points, `r_values` stores the position vectors, and `v_values` stores the velocity vectors.
   - The initial position $ r_0 $ and velocity $ v_0 $ are assigned to the first elements of `r_values` and `v_values`.

2. **Initial Energy Calculation**:
   - The initial specific mechanical energy of the system is calculated using the `total_energy` function and stored in `initial_energy`.
   - The energy error array `energy_errors` is initialized, with the error set to zero at the initial time step.

3. **Main Loop for Time Stepping**:
   - For each time step $ n $, Heun's method is applied using a predictor-corrector approach:
   
   **Predictor Step**:
   - The position and velocity are first predicted using the **Forward Euler method**:
     $$
     r_{\text{predictor}} = r_n + h \cdot v_n
     $$
     $$
     v_{\text{predictor}} = v_n + h \cdot a(r_n)
     $$
     where $ a(r_n) $ is the gravitational acceleration computed by the `gravity_acceleration` function.

   **Corrector Step**:
   - The final position and velocity are corrected by taking the average of the current and predicted values:
     $$
     r_{n+1} = r_n + \frac{h}{2} \cdot (v_n + v_{\text{predictor}})
     $$
     $$
     v_{n+1} = v_n + \frac{h}{2} \cdot (a(r_n) + a(r_{\text{predictor}}))
     $$
     This correction improves the accuracy by considering the change in both position and velocity over the time step.

4. **Energy Calculation**:
   - After updating the position and velocity, the specific mechanical energy at each step is calculated using the `total_energy` function. The energy error at each step is computed as the absolute difference between the current energy and the initial energy:
     $$
     \text{energy\_error}_{n+1} = |E_{n+1} - E_0|
     $$

5. **Final Output**:
   - The function returns the `t_values` (time points), `r_values` (position vectors), `v_values` (velocity vectors), and `energy_errors` (energy errors at each time point).

### Example Workflow:
1. **Predictor Step**: The method first predicts the next position and velocity using the Forward Euler method.
2. **Corrector Step**: The method then corrects these predictions by averaging the current and predicted values, providing second-order accuracy.
3. **Improved Energy Conservation**: Compared to the Forward Euler method, Heun’s method improves energy conservation, resulting in smaller energy errors.

### Key Advantages:
- **Higher Accuracy**: Heun’s method is more accurate than the Forward Euler method due to its second-order accuracy.
- **Predictor-Corrector Approach**: By using both a prediction and correction step, the method provides a better estimate of the next position and velocity.

### Limitations:
- **More Computational Effort**: Heun’s method requires more calculations than the Forward Euler method since it evaluates the function twice per time step.
- **Energy Conservation**: While improved, energy conservation is still not perfect, and energy errors may accumulate over long simulations.

In [ ]:
def rk2_heun_method(r0, v0, t0, t_end, h):
    """
    Solve the Kepler problem using Heun's method (explicit midpoint method).

    Parameters:
        r0 (np.ndarray): Initial position vector [x0, y0]
        v0 (np.ndarray): Initial velocity vector [vx0, vy0]
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size

    Returns:
        t_values (np.ndarray): Array of time points
        r_values (np.ndarray): Array of position vectors at each time point
        v_values (np.ndarray): Array of velocity vectors at each time point
        energy_errors (np.ndarray): Array of energy errors at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    
    r_values = np.zeros((N + 1, 2))  # Store position vectors [x, y]
    v_values = np.zeros((N + 1, 2))  # Store velocity vectors [vx, vy]
    
    r_values[0] = r0
    v_values[0] = v0

    # Calculate initial energy
    initial_energy = total_energy(r0, v0)
    energy_errors = np.zeros(N + 1)
    energy_errors[0] = 0  # Initial energy error is 0

    for n in range(N):
        r_n = r_values[n]
        v_n = v_values[n]
        a_n = gravity_acceleration(r_n)
        
        # Predictor step: Forward Euler for both r and v
        r_predictor = r_n + h * v_n
        v_predictor = v_n + h * a_n
        
        # Corrector step: using average of current and predicted values
        r_values[n + 1] = r_n + (h / 2) * (v_n + v_predictor)
        v_values[n + 1] = v_n + (h / 2) * (a_n + gravity_acceleration(r_predictor))

        # Calculate energy at this step
        energy_n = total_energy(r_values[n + 1], v_values[n + 1])
        energy_errors[n + 1] = abs(energy_n - initial_energy)  # Absolute error in energy

    return t_values, r_values, v_values, energy_errors


## Explanation of the `rk2_midpoint_method` Function

This function implements the **midpoint method**, which is a second-order **Runge-Kutta (RK2)** method, for solving the **Kepler problem**. The midpoint method improves upon the basic Euler method by using an intermediate point (midpoint) to compute a more accurate update for the position and velocity, and it tracks the energy errors to evaluate accuracy.

### Function Parameters:
- **`r0` (np.ndarray)**: The initial position vector $[x_0, y_0]$, representing the starting coordinates of the body.
- **`v0` (np.ndarray)**: The initial velocity vector $[v_{x0}, v_{y0}]$, representing the initial velocity components of the body.
- **`t0` (float)**: The starting time of the simulation.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization, determining the interval between updates.

### Returns:
- **`t_values` (np.ndarray)**: An array containing the time points from $ t_0 $ to $ t_{\text{end}} $.
- **`r_values` (np.ndarray)**: An array containing the position vectors $[x, y]$ at each time point.
- **`v_values` (np.ndarray)**: An array containing the velocity vectors $[v_x, v_y]$ at each time point.
- **`energy_errors` (np.ndarray)**: An array containing the errors in the specific mechanical energy at each time point, showing how well the method conserves energy.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps $ N $ is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, `r_values` stores the position vectors, and `v_values` stores the velocity vectors.
   - The initial position $ r_0 $ and velocity $ v_0 $ are assigned to the first elements of `r_values` and `v_values`.

2. **Initial Energy Calculation**:
   - The initial specific mechanical energy of the system is calculated using the `total_energy` function and stored in `initial_energy`.
   - The energy error array `energy_errors` is initialized, with the error set to zero at the initial time step.

3. **Main Loop for Time Stepping**:
   - For each time step $ n $, the midpoint method is applied to update the position and velocity:
   
   **Predictor Step (Euler's Step to Midpoint)**:
   - The position and velocity at the midpoint are first predicted using half a time step:
     $$
     r_{\text{mid}} = r_n + \frac{h}{2} \cdot v_n
     $$
     $$
     v_{\text{mid}} = v_n + \frac{h}{2} \cdot a(r_n)
     $$
     where $ a(r_n) $ is the gravitational acceleration computed by the `gravity_acceleration` function.

   **Corrector Step**:
   - The final position and velocity are corrected by using the midpoint values:
     $$
     r_{n+1} = r_n + h \cdot v_{\text{mid}}
     $$
     $$
     v_{n+1} = v_n + h \cdot a(r_{\text{mid}})
     $$
     This correction improves the accuracy of the position and velocity updates.

4. **Energy Calculation**:
   - After updating the position and velocity, the specific mechanical energy at each step is calculated using the `total_energy` function. The energy error at each step is computed as the absolute difference between the current energy and the initial energy:
     $$
     \text{energy\_error}_{n+1} = |E_{n+1} - E_0|
     $$

5. **Final Output**:
   - The function returns the `t_values` (time points), `r_values` (position vectors), `v_values` (velocity vectors), and `energy_errors` (energy errors at each time point).

### Example Workflow:
1. **Predictor Step**: The method predicts the midpoint values of position and velocity using half a time step.
2. **Corrector Step**: The method then corrects these predictions by using the midpoint values to compute the final position and velocity for the next time step.
3. **Improved Energy Conservation**: Compared to the Forward Euler method, the midpoint method provides better energy conservation and accuracy.

### Key Advantages:
- **Higher Accuracy**: The midpoint method is more accurate than the Forward Euler method due to its second-order accuracy.
- **Midpoint Correction**: By using the midpoint values to correct the solution, this method provides a better estimate of the next position and velocity.

### Limitations:
- **More Computational Effort**: The midpoint method requires more calculations than the Forward Euler method since it evaluates the function at the midpoint.
- **Energy Conservation**: While improved, energy conservation is still not perfect, and energy errors may accumulate over long simulations.

In [ ]:
def rk2_midpoint_method(r0, v0, t0, t_end, h):
    """
    Solve the Kepler problem using the midpoint method (explicit midpoint method).

    Parameters:
        r0 (np.ndarray): Initial position vector [x0, y0]
        v0 (np.ndarray): Initial velocity vector [vx0, vy0]
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size

    Returns:
        t_values (np.ndarray): Array of time points
        r_values (np.ndarray): Array of position vectors at each time point
        v_values (np.ndarray): Array of velocity vectors at each time point
        energy_errors (np.ndarray): Array of energy errors at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    
    r_values = np.zeros((N + 1, 2))  # Store position vectors [x, y]
    v_values = np.zeros((N + 1, 2))  # Store velocity vectors [vx, vy]
    
    r_values[0] = r0
    v_values[0] = v0

    # Calculate initial energy
    initial_energy = total_energy(r0, v0)
    energy_errors = np.zeros(N + 1)
    energy_errors[0] = 0  # Initial energy error is 0

    for n in range(N):
        r_n = r_values[n]
        v_n = v_values[n]
        
        # Predictor step: Euler's step to midpoint
        r_mid = r_n + (h / 2) * v_n
        v_mid = v_n + (h / 2) * gravity_acceleration(r_n)
        
        # Corrector step: Update position and velocity using the midpoint
        r_values[n + 1] = r_n + h * v_mid
        v_values[n + 1] = v_n + h * gravity_acceleration(r_mid)

        # Calculate energy at this step
        energy_n = total_energy(r_values[n + 1], v_values[n + 1])
        energy_errors[n + 1] = abs(energy_n - initial_energy)  # Absolute error in energy

    return t_values, r_values, v_values, energy_errors

In [ ]:
# Circular-orbit demonstration: compare position and energy errors separately.
compare_circular_orbit({'Euler': euler_method, 'Heun': rk2_heun_method, 'Midpoint': rk2_midpoint_method})


## Explanation of the `rk4_method` Function

This function implements the **4th-order Runge-Kutta (RK4) method** for solving the **Kepler problem**. The RK4 method is a widely used numerical integration technique that provides fourth-order accuracy. It is more accurate than both the Forward Euler and midpoint methods and is commonly used for solving differential equations where high precision is required. In this case, the method updates the position and velocity of a body under gravitational influence and tracks energy errors to assess how well energy is conserved.

### Function Parameters:
- **`r0` (np.ndarray)**: The initial position vector $[x_0, y_0]$, representing the starting coordinates of the body.
- **`v0` (np.ndarray)**: The initial velocity vector $[v_{x0}, v_{y0}]$, representing the initial velocity components of the body.
- **`t0` (float)**: The starting time of the simulation.
- **`t_end` (float)**: The end time of the simulation.
- **`h` (float)**: The step size for time discretization, determining the interval between updates.

### Returns:
- **`t_values` (np.ndarray)**: An array containing the time points from $ t_0 $ to $ t_{\text{end}} $.
- **`r_values` (np.ndarray)**: An array containing the position vectors $[x, y]$ at each time point.
- **`v_values` (np.ndarray)**: An array containing the velocity vectors $[v_x, v_y]$ at each time point.
- **`energy_errors` (np.ndarray)**: An array containing the errors in the specific mechanical energy at each time point, showing how well the method conserves energy.

### Key Steps in the Code:

1. **Time and Solution Arrays Initialization**:
   - The number of steps $ N $ is calculated based on the total simulation time and the step size.
   - The `t_values` array holds the time points, `r_values` stores the position vectors, and `v_values` stores the velocity vectors.
   - The initial position $ r_0 $ and velocity $ v_0 $ are assigned to the first elements of `r_values` and `v_values`.

2. **Initial Energy Calculation**:
   - The initial specific mechanical energy of the system is calculated using the `total_energy` function and stored in `initial_energy`.
   - The energy error array `energy_errors` is initialized, with the error set to zero at the initial time step.

3. **Main Loop for RK4 Time Stepping**:
   - For each time step $ n $, the RK4 method is used to update the position and velocity:
   
   **RK4 Steps**:
   - Four intermediate stages ($ k_1, k_2, k_3, k_4 $) are calculated to estimate the next position and velocity:
   
   - **Stage 1 (k1)**:
     $$
     k1_r = v_n
     $$
     $$
     k1_v = a(r_n)
     $$
     where $ a(r_n) $ is the gravitational acceleration at the current position $ r_n $.

   - **Stage 2 (k2)**:
     $$
     k2_r = v_n + 0.5 h \cdot k1_v
     $$
     $$
     k2_v = a(r_n + 0.5 h \cdot k1_r)
     $$
     where $ k2_r $ and $ k2_v $ are evaluated at the midpoint.

   - **Stage 3 (k3)**:
     $$
     k3_r = v_n + 0.5 h \cdot k2_v
     $$
     $$
     k3_v = a(r_n + 0.5 h \cdot k2_r)
     $$
     where $ k3_r $ and $ k3_v $ refine the values further using the midpoint.

   - **Stage 4 (k4)**:
     $$
     k4_r = v_n + h \cdot k3_v
     $$
     $$
     k4_v = a(r_n + h \cdot k3_r)
     $$
     where $ k4_r $ and $ k4_v $ estimate the values at the next time step.

   **Position and Velocity Update**:
   - The next position $ r_{n+1} $ and velocity $ v_{n+1} $ are calculated as a weighted sum of these four stages:
     $$
     r_{n+1} = r_n + \frac{h}{6} (k1_r + 2 \cdot k2_r + 2 \cdot k3_r + k4_r)
     $$
     $$
     v_{n+1} = v_n + \frac{h}{6} (k1_v + 2 \cdot k2_v + 2 \cdot k3_v + k4_v)
     $$

4. **Energy Calculation**:
   - After updating the position and velocity, the specific mechanical energy at each step is calculated using the `total_energy` function. The energy error at each step is computed as the absolute difference between the current energy and the initial energy:
     $$
     \text{energy\_error}_{n+1} = |E_{n+1} - E_0|
     $$

5. **Final Output**:
   - The function returns the `t_values` (time points), `r_values` (position vectors), `v_values` (velocity vectors), and `energy_errors` (energy errors at each time point).

### Example Workflow:
1. **RK4 Stages**: Four intermediate stages are calculated using the position and velocity at different time steps to refine the accuracy of the update.
2. **Energy Conservation**: The RK4 method provides much better energy conservation than simpler methods like Forward Euler due to its fourth-order accuracy.
3. **Kepler Problem Simulation**: The RK4 method accurately simulates the motion of a body in a gravitational field, providing highly precise position and velocity updates.

### Key Advantages:
- **High Accuracy**: The RK4 method is highly accurate, with fourth-order convergence, making it one of the most commonly used numerical methods for ODEs.
- **Stable Energy Conservation**: Compared to lower-order methods, RK4 conserves energy much more effectively, reducing the energy errors over time.

### Limitations:
- **More Computational Effort**: The RK4 method requires more function evaluations (four per time step), making it computationally more expensive than lower-order methods.
- **Still Not Perfect**: Despite its high accuracy, energy errors can still accumulate over very long simulations, especially if the step size $ h $ is not small enough.​⬤

In [ ]:
def rk4_method(r0, v0, t0, t_end, h):
    """
    Solve the Kepler problem using the 4th-order Runge-Kutta method (RK4).
    
    Parameters:
        r0 (numpy.ndarray): Initial position vector [x, y].
        v0 (numpy.ndarray): Initial velocity vector [vx, vy].
        t0 (float): Initial time.
        t_end (float): Final time.
        h (float): Time step size.

    Returns:
        t_values (numpy.ndarray): Array of time points.
        r_values (numpy.ndarray): Array of position vectors at each time point.
        v_values (numpy.ndarray): Array of velocity vectors at each time point.
        energy_errors (numpy.ndarray): Array of energy errors at each time point.
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    r_values = np.zeros((N + 1, 2))
    v_values = np.zeros((N + 1, 2))
    energy_errors = np.zeros(N + 1)

    # Initial conditions
    r_values[0] = r0
    v_values[0] = v0
    initial_energy = total_energy(r0, v0)

    # RK4 integration
    for n in range(N):
        t_n = t_values[n]
        r_n = r_values[n]
        v_n = v_values[n]

        # Calculate k1 values
        k1_r = v_n
        k1_v = gravity_acceleration(r_n)

        # Calculate k2 values
        k2_r = v_n + 0.5 * h * k1_v
        k2_v = gravity_acceleration(r_n + 0.5 * h * k1_r)

        # Calculate k3 values
        k3_r = v_n + 0.5 * h * k2_v
        k3_v = gravity_acceleration(r_n + 0.5 * h * k2_r)

        # Calculate k4 values
        k4_r = v_n + h * k3_v
        k4_v = gravity_acceleration(r_n + h * k3_r)

        # Update position and velocity
        r_new = r_n + (h / 6.0) * (k1_r + 2 * k2_r + 2 * k3_r + k4_r)
        v_new = v_n + (h / 6.0) * (k1_v + 2 * k2_v + 2 * k3_v + k4_v)

        r_values[n + 1] = r_new
        v_values[n + 1] = v_new

        # Calculate energy error at this step
        current_energy = total_energy(r_new, v_new)
        energy_errors[n + 1] = np.abs(current_energy - initial_energy)

    return t_values, r_values, v_values, energy_errors

In [ ]:
# Circular-orbit demonstration: compare position and energy errors separately.
compare_circular_orbit({'Euler': euler_method, 'Heun': rk2_heun_method, 'Midpoint': rk2_midpoint_method, 'RK4': rk4_method})


## Kepler問題に対する2次Adams予測修正法

状態 $\mathbf y=(\mathbf r,\mathbf v)$、右辺 $\mathbf f=(\mathbf v,\mathbf a(\mathbf r))$ に、[ex04](ex04%20adams_accuracy.ipynb) のAB2–AM2更新式を適用します。RK4で1ステップの開始値を求め、予測後の修正は1回です。

通常の大域誤差は2次です。過去の加速度を再評価する実装なので、開始計算を除いて1ステップ3回の加速度評価を行います。シンプレクティック法ではなく、エネルギーの厳密保存を保証しません。


In [ ]:
def adams_bashforth_moulton_2nd(r0, v0, t0, t_end, h):
    """
    Solve the Kepler problem using the 2nd-order Adams-Bashforth-Moulton predictor-corrector method.

    Parameters:
        r0 (array): Initial position vector
        v0 (array): Initial velocity vector
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size

    Returns:
        t_values (numpy.ndarray): Array of time points
        r_values (numpy.ndarray): Array of position vectors at each time point
        v_values (numpy.ndarray): Array of velocity vectors at each time point
        energy_errors (list): List of energy errors at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    r_values = np.zeros((N + 1, 2))  # For 2D position vectors
    v_values = np.zeros((N + 1, 2))  # For 2D velocity vectors
    r_values[0] = r0
    v_values[0] = v0

    # Start with RK4 to get the second point (since Adams methods require two initial points)
    t_values_rk4, r_values_rk4, v_values_rk4, _ = rk4_method(r0, v0, t0, t0 + h, h)
    r_values[1] = r_values_rk4[-1]
    v_values[1] = v_values_rk4[-1]

    energy_errors = np.zeros(N + 1)
    energy_errors[1] = abs(total_energy(r_values[1], v_values[1]) - total_energy(r0, v0))

    for n in range(1, N):
        t_n = t_values[n]
        t_np1 = t_values[n + 1]  # t_n+1
        r_n = r_values[n]
        v_n = v_values[n]
        r_nm1 = r_values[n - 1]
        v_nm1 = v_values[n - 1]

        # Predictor step (Adams-Bashforth 2nd order)
        a_n = gravity_acceleration(r_n)
        a_nm1 = gravity_acceleration(r_nm1)
        v_predictor = v_n + (h / 2) * (3 * a_n - a_nm1)
        r_predictor = r_n + (h / 2) * (3 * v_n - v_nm1)

        # Corrector step (Adams-Moulton 2nd order)
        a_predictor = gravity_acceleration(r_predictor)
        v_corrector = v_n + (h / 2) * (a_predictor + a_n)
        r_corrector = r_n + (h / 2) * (v_predictor + v_n)

        # Update position and velocity
        r_values[n + 1] = r_corrector
        v_values[n + 1] = v_corrector

        # Calculate energy error
        current_energy = total_energy(r_values[n + 1], v_values[n + 1])
        initial_energy = total_energy(r0, v0)
        energy_errors[n + 1] = abs(initial_energy - current_energy)

    return t_values, r_values, v_values, energy_errors

In [ ]:
# Circular-orbit demonstration: compare position and energy errors separately.
compare_circular_orbit({'Euler': euler_method, 'Heun': rk2_heun_method, 'Midpoint': rk2_midpoint_method, 'RK4': rk4_method, 'Adams2': adams_bashforth_moulton_2nd})


## Kepler問題に対する4次Adams予測修正法

状態 $\mathbf y=(\mathbf r,\mathbf v)$ に、[ex04](ex04%20adams_accuracy.ipynb) のAB4–AM4更新式を適用します。RK4で3ステップ進めて開始値を作り、予測後の修正は1回です。

通常の大域誤差は4次です。過去の加速度を再評価するため、開始計算を除いて1ステップ5回の加速度評価を行います。RK4より速いとは限りません。安定性や長時間のエネルギー誤差は次数だけからは判断できません。


In [ ]:
def adams_bashforth_moulton_4th(r0, v0, t0, t_end, h):
    """
    Solve the Kepler problem using the 4th-order Adams-Bashforth-Moulton predictor-corrector method.
    
    Parameters:
        r0 (numpy.ndarray): Initial position vector [x, y]
        v0 (numpy.ndarray): Initial velocity vector [vx, vy]
        t0 (float): Initial time
        t_end (float): End time
        h (float): Time step size
    
    Returns:
        tuple: (t_values, r_values, v_values, energy_errors)
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    r_values = np.zeros((N + 1, 2))  # Position vectors
    v_values = np.zeros((N + 1, 2))  # Velocity vectors
    energy_errors = np.zeros(N + 1)
    
    # Initial conditions
    r_values[0] = r0
    v_values[0] = v0
    initial_energy = total_energy(r0, v0)
    energy_errors[0] = 0.0
    
    # Use RK4 to initialize the first three steps
    for i in range(min(3, N)):
        t_start = t0 + i * h
        t_end_rk4 = t_start + h
        _, r_rk4, v_rk4, _ = rk4_method(r_values[i], v_values[i], t_start, t_end_rk4, h)
        r_values[i + 1] = r_rk4[-1]
        v_values[i + 1] = v_rk4[-1]
        # Calculate energy error for the initialized points
        energy_errors[i + 1] = np.abs(total_energy(r_values[i + 1], v_values[i + 1]) - initial_energy)
    
    for n in range(3, N):
        r_n = r_values[n]
        v_n = v_values[n]
        r_nm1 = r_values[n - 1]
        v_nm1 = v_values[n - 1]
        r_nm2 = r_values[n - 2]
        v_nm2 = v_values[n - 2]
        r_nm3 = r_values[n - 3]
        v_nm3 = v_values[n - 3]
        
        # Get accelerations
        a_n   = gravity_acceleration(r_n)
        a_nm1 = gravity_acceleration(r_nm1)
        a_nm2 = gravity_acceleration(r_nm2)
        a_nm3 = gravity_acceleration(r_nm3)
        
        # Predictor step (Adams-Bashforth 4th order)
        r_predictor = r_n + (h / 24) * (55 * v_n - 59 * v_nm1 + 37 * v_nm2 - 9 * v_nm3)
        v_predictor = v_n + (h / 24) * (55 * a_n - 59 * a_nm1 + 37 * a_nm2 - 9 * a_nm3)
        
        # Compute acceleration at predicted position
        a_predictor = gravity_acceleration(r_predictor)
        
        # Corrector step (Adams-Moulton 4th order)
        r_corrector = r_n + (h / 24) * (9 * v_predictor + 19 * v_n - 5 * v_nm1 + v_nm2)
        v_corrector = v_n + (h / 24) * (9 * a_predictor + 19 * a_n - 5 * a_nm1 + a_nm2)
        
        # Update the solution
        r_values[n + 1] = r_corrector
        v_values[n + 1] = v_corrector
        
        # Calculate energy error
        current_energy = total_energy(r_corrector, v_corrector)
        energy_errors[n + 1] = np.abs(current_energy - initial_energy)
    
    return t_values, r_values, v_values, energy_errors


In [ ]:
# Circular-orbit demonstration: compare position and energy errors separately.
compare_circular_orbit({'Euler': euler_method, 'Heun': rk2_heun_method, 'Midpoint': rk2_midpoint_method, 'RK4': rk4_method, 'Adams2': adams_bashforth_moulton_2nd, 'Adams4': adams_bashforth_moulton_4th})


## Leapfrog（kick–drift–kick）法

この実装は固定刻みの2次精度のKDK法です。
$$\mathbf v_{n+1/2}=\mathbf v_n+\frac h2\mathbf a(\mathbf r_n),$$
$$\mathbf r_{n+1}=\mathbf r_n+h\mathbf v_{n+1/2},$$
$$\mathbf v_{n+1}=\mathbf v_{n+1/2}+\frac h2\mathbf a(\mathbf r_{n+1}).$$

分離可能なHamiltonianに対し、シンプレクティック構造と時間対称性を保ちます。これは元のHamiltonian（エネルギー）の厳密保存を意味しません。衝突特異点から離れた滑らかな軌道を十分小さい固定刻みで積分する場合、後退誤差解析で修正Hamiltonianに近い保存量を考えられ、元のエネルギー誤差は長時間小さく振動する傾向があります。任意の刻みや任意の時間での有界性を保証するものではありません。

この中心力問題では各kickとdriftが角運動量を保存するため、その誤差は通常丸め誤差程度です。エネルギー誤差と位置・位相誤差は別の指標です。円軌道のエネルギー誤差に4次に近い傾きが見えても、Leapfrogの軌道精度は2次です。

参考：[Hairer, Lubich & Wanner, Störmer/Verlet法の幾何学的性質](https://www.unige.ch/~hairer/preprints/gniverlet.html)。


In [ ]:
def leapfrog_method(r0, v0, t0, t_end, h):
    """
    Solve the Kepler problem using the Leapfrog method.

    Parameters:
        r0 (np.ndarray): Initial position vector [x0, y0]
        v0 (np.ndarray): Initial velocity vector [vx0, vy0]
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size

    Returns:
        t_values (np.ndarray): Array of time points
        r_values (np.ndarray): Array of position vectors at each time point
        v_values (np.ndarray): Array of velocity vectors at each time point
        energy_errors (np.ndarray): Array of energy errors at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    
    r_values = np.zeros((N + 1, 2))  # Store position vectors [x, y]
    v_values = np.zeros((N + 1, 2))  # Store velocity vectors [vx, vy]
    a_values = np.zeros((N + 1, 2))  # Store velocity vectors [ax, ay]
    
    r_values[0] = r0
    v_values[0] = v0
    a_values[0] = gravity_acceleration(r0)

    # Calculate initial energy
    initial_energy = total_energy(r0, v0)
    energy_errors = np.zeros(N + 1)
    energy_errors[0] = 0  # Initial energy error is 0

    for n in range(N):
        # kick
        v_n = v_values[n]
        v_half = v_n + 0.5 * h * a_values[n]

        # drift
        r_n = r_values[n]
        r_values[n + 1] = r_n + h * v_half

        # kick
        a_values[n + 1] = gravity_acceleration(r_values[n + 1])
        v_values[n + 1] = v_half + 0.5 * h * a_values[n + 1]

        # Calculate energy at this step
        energy_n = total_energy(r_values[n + 1], v_values[n + 1])
        energy_errors[n + 1] = abs(energy_n - initial_energy)  # Absolute error in energy

    return t_values, r_values, v_values, energy_errors

In [ ]:
# Circular-orbit demonstration: compare position and energy errors separately.
compare_circular_orbit({'Euler': euler_method, 'Heun': rk2_heun_method, 'Midpoint': rk2_midpoint_method, 'RK4': rk4_method, 'Adams2': adams_bashforth_moulton_2nd, 'Adams4': adams_bashforth_moulton_4th, 'Leapfrog': leapfrog_method})


## 長時間積分：楕円軌道（初速度0.6、刻み0.001、通常の終了時刻1000）

各手法で比エネルギーの絶対誤差と計算時間を比較します。Leapfrogもエネルギーを厳密保存するわけではありません。時間刻みが軌道を十分に分解していることを確認してください。実行時間は環境に依存します。

冒頭の `QUICK_DEMO = True` で終了時刻だけを1/10にできます。既定値は `False` で、元の長時間計算を行います。軽量版は長時間挙動の検証を置き換えるものではありません。


In [ ]:
def main():
    # Initial conditions for the Kepler problem
    r0 = np.array([1.0, 0.0])  # Initial position
    v0 = np.array([0.0, 0.6])  # Initial velocity (perpendicular)
    
    t0 = 0.0
    t_end = 1.e+2 if QUICK_DEMO else 1.e+3
    h = 0.001  # Step size for time evolution plots

    # Add Leapfrog and other methods to the comparison
    methods = {
        'Euler': euler_method,
        'RK2 Heun': rk2_heun_method,
        'RK2 Midpoint': rk2_midpoint_method,
        'RK4': rk4_method,
        'Adams2': adams_bashforth_moulton_2nd,
        'Adams4': adams_bashforth_moulton_4th,
        'Leapfrog': leapfrog_method  # Adding Leapfrog method
    }
    
    colors = {
        'Euler': '#000000',
        'RK2 Heun': '#E69F00',
        'RK2 Midpoint': '#56B4E9',
        'RK4': '#009E73',
        'Adams2': '#0072B2',
        'Adams4': '#D55E00',
        'Leapfrog': '#CC79A7'  # Color for Leapfrog method
    }

    # Dictionary to store computation times
    computation_times = {}

    # Plot energy errors over time for the chosen h (with log-log scale for both axes)
    size = 10
    plt.figure(figsize=(2*size, size))

    for method_name, method in methods.items():
        # Start timing
        start_time = time.perf_counter()

        t_values, _, _, energy_errors = method(r0, v0, t0, t_end, h)
        
        # End timing
        elapsed_time = time.perf_counter() - start_time
        computation_times[method_name] = elapsed_time

        assert len(t_values) == len(energy_errors)
        
        # Plot the energy errors
        plt.loglog(t_values, energy_errors, lw=3, **{k: v for k, v in METHOD_STYLES[method_name.replace('RK2 ', '')].items() if k != 'marker'}, label=f'{method_name} Energy Error')
    
    plt.xlabel('Time')
    plt.ylabel('Absolute Specific Energy Error')
    plt.title(f'Energy Error over Time (h={h})')
    plt.grid(True)

    # Adjust legend to be placed outside the plot (on the right)
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', borderaxespad=0.)

    plt.tight_layout()
    plt.show()

    # Print computation times
    print("\nComputation Times (seconds):")
    for method_name, time_elapsed in computation_times.items():
        print(f"{method_name}: {time_elapsed:.4f} s")

if __name__ == "__main__":
    main()


## 長時間積分：円軌道（初速度1、刻み0.01、通常の終了時刻10000）

各手法で比エネルギーの絶対誤差と計算時間を比較します。Leapfrogもエネルギーを厳密保存するわけではありません。時間刻みが軌道を十分に分解していることを確認してください。実行時間は環境に依存します。

冒頭の `QUICK_DEMO = True` で終了時刻だけを1/10にできます。既定値は `False` で、元の長時間計算を行います。軽量版は長時間挙動の検証を置き換えるものではありません。


In [ ]:
def main():
    # Initial conditions for the Kepler problem
    r0 = np.array([1.0, 0.0])  # Initial position
    v0 = np.array([0.0, 1.0])  # Initial velocity (perpendicular)
    
    t0 = 0.0
    t_end = 1.e+3 if QUICK_DEMO else 1.e+4
    h = 0.01  # Step size for time evolution plots

    # Add Leapfrog and other methods to the comparison
    methods = {
        'Euler': euler_method,
        'RK2 Heun': rk2_heun_method,
        'RK2 Midpoint': rk2_midpoint_method,
        'RK4': rk4_method,
        'Adams2': adams_bashforth_moulton_2nd,
        'Adams4': adams_bashforth_moulton_4th,
        'Leapfrog': leapfrog_method 
    }
    
    colors = {
        'Euler': '#000000',
        'RK2 Heun': '#E69F00',
        'RK2 Midpoint': '#56B4E9',
        'RK4': '#009E73',
        'Adams2': '#0072B2',
        'Adams4': '#D55E00',
        'Leapfrog': '#CC79A7'
    }

    # Dictionary to store computation times
    computation_times = {}

    # Create figure with two subplots
    size = 10
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2*size, size))

    # Plot energy errors over time for the chosen h (with log-log scale for both axes)
    for method_name, method in methods.items():
        # Start timing
        start_time = time.perf_counter()

        t_values, _, _, energy_errors = method(r0, v0, t0, t_end, h)
        
        # End timing
        elapsed_time = time.perf_counter() - start_time
        computation_times[method_name] = elapsed_time

        assert len(t_values) == len(energy_errors)
        
        # Plot the energy errors
        ax1.loglog(t_values, energy_errors, lw=3, **{k: v for k, v in METHOD_STYLES[method_name.replace('RK2 ', '')].items() if k != 'marker'}, label=f'{method_name} Energy Error')

    # Customize first plot
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Absolute Specific Energy Error')
    ax1.set_title(f'Energy Error over Time (dt={h})')
    ax1.grid(True)

    # Prepare data for the horizontal bar chart (computation times)
    method_names = list(computation_times.keys())
    times = list(computation_times.values())
    
    # Plot horizontal bar chart for computation times
    ax2.barh(method_names, times, color=[colors[m] for m in method_names])
    #ax2.set_ylabel('Methods')
    ax2.set_xlabel('Computation Time (seconds)')
    ax2.set_title('Computation Time Comparison')
    ax2.grid(True)

    # Adjust layout
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()
